In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install git+https://github.com/neurallatents/nlb_tools.git
!pip install dandi
!dandi download https://gui.dandiarchive.org/dandiset/000127
!pip install git+https://github.com/neurallatents/nlb_tools.git --no-deps
!pip install pynwb elephant pyyaml

  Cloning https://github.com/neurallatents/nlb_tools.git to /tmp/pip-req-build-vbnr8onz
  Running command git clone --filter=blob:none --quiet https://github.com/neurallatents/nlb_tools.git /tmp/pip-req-build-vbnr8onz
  Resolved https://github.com/neurallatents/nlb_tools.git to commit 42f8410b88e12db136910fa2f888b025ea0aa2ae
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 92.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See abov

## Area2_Bump Exploration.

Full exploration of the Area2_Bump primate dataset (DANDI 000127):
  1. Load + resample + smooth
  2. Enumerate trial types and behavioral geometry
  3. Trialize around move_onset_time  (window: -100..+500 ms)
  4. Per-condition PSTHs (16 conditions x 120 bins x 65 neurons)
  5. Per-neuron tuning curves (active trials, von Mises fit)
  6. PCA neural trajectories (active/passive fit separately)
  7. Ridge decoder of hand_vel (active and passive blocks)
  8. Save canonical tensors + figures to OUT_DIR.

**Outputs:**
  - area2_explore_summary.json    — machine-readable summary of every stage
  - figures/                      — PNGs for each plot
  - area2_bump_canonical.h5       — stacked tensors for downstream sub-tasks

In [3]:
from __future__ import annotations

import os
import sys
import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Union

import numpy as np
import pandas as pd
import h5py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import curve_fit
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold

from nlb_tools.nwb_interface import NWBDataset

# CORE COMPONENTS: CONFIG, DATASET, TRIALIZER, PSTH, DECODER, PLOTS

# CONFIG
@dataclass
class ExploreConfig:
    """All knobs in one place. Mirrors configs/area2_bump.yaml."""
    raw_dir: Union[str, Path] = "/kaggle/working/000127/sub-Han"
    out_dir: Union[str, Path] = "/kaggle/working"

    # Preprocessing
    train_prefix: str = "*train"
    split_heldout: bool = False
    bin_width_ms: int = 5
    smooth_sd_ms: int = 40
    smooth_name: str = "smth_40"

    # Trialization
    align_field: str = "move_onset_time"
    align_range_ms: Tuple[int, int] = (-100, 500)
    allow_overlap: bool = False
    allow_nans: bool = False
    split_valid_only: bool = True

    # PSTH
    min_trials_per_condition: int = 5

    # Tuning
    tuning_window_ms: Tuple[int, int] = (0, 500)

    # PCA
    pca_n_components: int = 3
    pca_standardize: bool = True
    pca_separate_active_passive: bool = True

    # Decoder
    decoder_lag_ms: int = 40
    decoder_alphas: List[float] = field(default_factory=lambda: [1e-4, 1e-3, 1e-2, 1e-1, 1.0])
    decoder_cv_folds: int = 5

    # Plot
    neuron_to_plot: int = 0
    max_raster_trials: int = 30

    def resolved_paths(self) -> Dict[str, Path]:
        out = Path(self.out_dir)
        fig = out / "figures"
        out.mkdir(parents=True, exist_ok=True)
        fig.mkdir(parents=True, exist_ok=True)
        return {
            "raw_dir": Path(self.raw_dir),
            "out_dir": out,
            "fig_dir": fig,
            "summary_json": out / "area2_explore_summary.json",
            "canonical_h5": out / "area2_bump_canonical.h5",
        }


# DATASET
class Area2BumpDataset:
    """Load NWB, resample to 5 ms, add smoothed rate field."""

    def __init__(self, cfg: ExploreConfig):
        self.cfg = cfg
        paths = cfg.resolved_paths()
        if not paths["raw_dir"].exists():
            raise FileNotFoundError(
                f"Raw data not found at {paths['raw_dir']}. "
                f"Run `dandi download https://dandiarchive.org/dandiset/000127/0.220113.0359`."
            )
        self.dataset = NWBDataset(
            fpath=str(paths["raw_dir"]),
            prefix=cfg.train_prefix,
            split_heldout=cfg.split_heldout,
            skip_fields=[],
        )
        self.dataset.resample(cfg.bin_width_ms)
        self.dataset.smooth_spk(cfg.smooth_sd_ms, name=cfg.smooth_name)

    @property
    def rate_field(self) -> str:
        return f"spikes_{self.cfg.smooth_name}"

    @property
    def n_neurons(self) -> int:
        return int(self.dataset.data["spikes"].shape[1])

    @property
    def n_trials(self) -> int:
        return int(len(self.dataset.trial_info))


# TRIAL TYPE TABLE
def make_trial_type_labels(trial_info: pd.DataFrame) -> pd.Series:
    labels = pd.Series(index=trial_info.index, dtype="object")
    has_bump = trial_info["ctr_hold_bump"].fillna(False).astype(bool)
    cond_dir = trial_info["cond_dir"]
    active = ~has_bump & cond_dir.notna()
    passive = has_bump & cond_dir.notna()
    labels[active] = "active_" + cond_dir[active].astype(int).astype(str).str.zfill(3)
    labels[passive] = "passive_" + cond_dir[passive].astype(int).astype(str).str.zfill(3)
    return labels


def get_trial_type_table(trial_info: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for bump in [False, True]:
        for d in sorted(trial_info["cond_dir"].dropna().unique()):
            mask = (trial_info["ctr_hold_bump"].fillna(False).astype(bool) == bump) & (trial_info["cond_dir"] == d)
            sub = trial_info[mask]
            if len(sub) == 0:
                continue
            r = sub["result"].value_counts().to_dict()
            s = sub["split"].value_counts().to_dict()
            rows.append({
                "trial_type_label": ("passive_" if bump else "active_") + f"{int(d):03d}",
                "ctr_hold_bump": bool(bump),
                "cond_dir": int(d),
                "n_total": int(len(sub)),
                "n_R": int(r.get("R", 0)),
                "n_A": int(r.get("A", 0)),
                "n_I": int(r.get("I", 0)),
                "n_F": int(r.get("F", 0)),
                "n_train": int(s.get("train", 0)),
                "n_val": int(s.get("val", 0)),
                "n_none": int(s.get("none", 0)),
            })
    return pd.DataFrame(rows).sort_values(["ctr_hold_bump", "cond_dir"]).reset_index(drop=True)


# TRIALIZED DATA CONTAINER
@dataclass
class TrializedData:
    """Stacked (n_trials, n_time, n_channels) arrays after trialization."""
    spikes: np.ndarray
    rates: np.ndarray
    hand_vel: Optional[np.ndarray]
    trial_info: pd.DataFrame
    align_field: str
    align_range_ms: Tuple[int, int]
    bin_width_ms: int
    time_bins_ms: np.ndarray

    @property
    def n_trials(self): return self.spikes.shape[0]
    @property
    def n_time(self): return self.spikes.shape[1]
    @property
    def n_neurons(self): return self.spikes.shape[2]


def trialize_area2_bump(a2: Area2BumpDataset, cfg: ExploreConfig) -> TrializedData:
    """Wrap NWBDataset.make_trial_data and stack into numpy arrays."""
    ti = a2.dataset.trial_info
    keep = pd.Series(True, index=ti.index)
    if cfg.split_valid_only:
        keep &= ti["split"].astype(str) != "none"
    ignored = ~keep

    trial_data = a2.dataset.make_trial_data(
        align_field=cfg.align_field,
        align_range=cfg.align_range_ms,
        ignored_trials=ignored,
        allow_overlap=cfg.allow_overlap,
        allow_nans=cfg.allow_nans,
    )
    if trial_data is None or len(trial_data) == 0:
        raise RuntimeError("Trialization produced no data.")

    spikes = _stack_field(trial_data, a2, "spikes").astype(np.int32)
    rates = _stack_field(trial_data, a2, a2.rate_field).astype(np.float32)
    hand_vel = _stack_field(trial_data, a2, "hand_vel")
    hand_vel = hand_vel.astype(np.float32) if hand_vel is not None else None

    surviving_ids = pd.Index(trial_data["trial_id"].unique())
    surviving_ti = a2.dataset.trial_info.set_index("trial_id").loc[surviving_ids].reset_index()

    t_start, t_end = cfg.align_range_ms
    time_bins_ms = np.linspace(t_start, t_end, spikes.shape[1], endpoint=False).astype(int)

    return TrializedData(
        spikes=spikes, rates=rates, hand_vel=hand_vel, trial_info=surviving_ti,
        align_field=cfg.align_field, align_range_ms=cfg.align_range_ms,
        bin_width_ms=a2.dataset.bin_width, time_bins_ms=time_bins_ms,
    )


def _stack_field(trial_data: pd.DataFrame, a2: Area2BumpDataset, field: str) -> Optional[np.ndarray]:
    top_cols = a2.dataset.data.columns.get_level_values(0).unique()
    if field not in top_cols:
        return None
    sub = trial_data[["trial_id", "align_time", field]].copy()
    sub = sub.sort_values(["trial_id", "align_time"])
    arrays = []
    expected_len = None
    for _, g in sub.groupby("trial_id", sort=False):
        arr = g[field].to_numpy()
        if expected_len is None:
            expected_len = arr.shape[0]
        elif arr.shape[0] != expected_len:
            if arr.shape[0] < expected_len:
                pad = np.zeros((expected_len - arr.shape[0],) + arr.shape[1:], dtype=arr.dtype)
                arr = np.concatenate([arr, pad], axis=0)
            else:
                arr = arr[:expected_len]
        arrays.append(arr)
    return np.stack(arrays, axis=0)


# PSTH
@dataclass
class PSTHResult:
    psth: np.ndarray
    condition_labels: List[str]
    condition_table: pd.DataFrame
    time_bins_ms: np.ndarray
    kern_sd_ms: int

    @property
    def n_conditions(self): return self.psth.shape[0]
    @property
    def n_time(self): return self.psth.shape[1]
    @property
    def n_neurons(self): return self.psth.shape[2]


def compute_psth(trialized: TrializedData, cfg: ExploreConfig) -> PSTHResult:
    """Trial-averaged firing rates per (ctr_hold_bump, cond_dir)."""
    ti = trialized.trial_info.copy()
    ti["trial_type_label"] = make_trial_type_labels(ti)
    kern_sd_bins = cfg.smooth_sd_ms / trialized.bin_width_ms

    # Re-smooth from raw spikes
    smoothed = gaussian_filter1d(
        trialized.spikes.astype(np.float32), sigma=kern_sd_bins, axis=1, mode="nearest"
    )
    id_col = "trial_id" if "trial_id" in ti.columns else "index"

    trial_ids = ti[id_col].to_numpy()

    labels_per_trial = (
        ti.set_index(id_col)["trial_type_label"]
          .reindex(trial_ids)
          .to_numpy()
    )
    unique_labels = sorted([l for l in np.unique(labels_per_trial) if isinstance(l, str)])

    psth_list, keep_labels, keep_meta = [], [], []
    for label in unique_labels:
        mask = labels_per_trial == label
        n = int(mask.sum())
        if n < cfg.min_trials_per_condition:
            continue
        psth_list.append(smoothed[mask].mean(axis=0))
        keep_labels.append(label)
        is_passive = label.startswith("passive")
        keep_meta.append({
            "trial_type_label": label,
            "ctr_hold_bump": is_passive,
            "cond_dir": int(label.split("_")[1]),
            "n_trials": n,
        })
    psth = np.stack(psth_list, axis=0)
    return PSTHResult(
        psth=psth, condition_labels=keep_labels,
        condition_table=pd.DataFrame(keep_meta),
        time_bins_ms=trialized.time_bins_ms, kern_sd_ms=cfg.smooth_sd_ms,
    )


# TUNING CURVES
def _von_mises(deg, baseline, amplitude, peak, kappa):
    rad = np.deg2rad(deg - peak)
    return baseline + amplitude * np.exp(kappa * (np.cos(rad) - 1.0))


def compute_tuning_curves(psth: PSTHResult, cfg: ExploreConfig) -> Dict:
    """Per-neuron direction tuning (active trials only): argmax + von Mises fit."""
    cond = psth.condition_table
    active_mask = ~cond["ctr_hold_bump"].astype(bool)
    active = cond[active_mask].reset_index(drop=True)
    active_idx = active.index.to_numpy()
    psth_active = psth.psth[active_idx]
    directions = active["cond_dir"].to_numpy()

    t = psth.time_bins_ms
    t0, t1 = cfg.tuning_window_ms
    tmask = (t >= t0) & (t <= t1)
    windowed = psth_active[:, tmask, :].mean(axis=1)

    sort_idx = np.argsort(directions)
    dirs_sorted = directions[sort_idx]
    tuning = windowed[sort_idx, :]
    n_neurons = tuning.shape[1]

    preferred = dirs_sorted[np.argmax(tuning, axis=0)]
    vm_peak = np.full(n_neurons, np.nan)
    vm_kappa = np.full(n_neurons, np.nan)
    fit_ok = np.zeros(n_neurons, dtype=bool)
    for n in range(n_neurons):
        rates = tuning[:, n]
        try:
            p0 = [float(rates.min()), float(rates.max() - rates.min()), float(preferred[n]), 2.0]
            bounds = ([0, 0, 0, 0.1], [np.inf, np.inf, 360, 20])
            popt, _ = curve_fit(_von_mises, dirs_sorted.astype(float), rates, p0=p0, bounds=bounds, maxfev=2000)
            vm_peak[n] = popt[2] % 360
            vm_kappa[n] = popt[3]
            fit_ok[n] = True
        except (RuntimeError, ValueError):
            vm_peak[n] = preferred[n]
            vm_kappa[n] = 0.0

    return {
        "directions_deg": dirs_sorted,
        "tuning_curves": tuning.T,
        "preferred_direction_argmax": preferred,
        "preferred_direction_vm": vm_peak,
        "von_mises_kappa": vm_kappa,
        "fit_success": fit_ok,
    }


# PCA TRAJECTORIES
def compute_pca_trajectories(psth: PSTHResult, cfg: ExploreConfig) -> Dict:
    """PCA on PSTHs, fit separately for active and passive blocks."""
    cond = psth.condition_table.reset_index(drop=True)
    n_conditions, n_time, n_neurons = psth.psth.shape
    trajectories = np.zeros((n_conditions, n_time, cfg.pca_n_components), dtype=np.float32)
    out = {}

    def _fit(idx):
        block = psth.psth[idx].reshape(-1, n_neurons)
        if cfg.pca_standardize:
            block = StandardScaler().fit_transform(block)
        pca = PCA(n_components=cfg.pca_n_components)
        traj = pca.fit_transform(block).reshape(len(idx), n_time, cfg.pca_n_components)
        return traj.astype(np.float32), pca.explained_variance_ratio_.astype(np.float32)

    if cfg.pca_separate_active_passive:
        for name, mask in [("active", ~cond["ctr_hold_bump"].astype(bool)),
                            ("passive", cond["ctr_hold_bump"].astype(bool))]:
            idx = np.where(mask)[0]
            if len(idx) == 0:
                continue
            traj, vr = _fit(idx)
            trajectories[idx] = traj
            out[name] = {"trajectories": traj, "explained_variance_ratio": vr, "indices": idx}
    else:
        idx = np.arange(n_conditions)
        traj, vr = _fit(idx)
        trajectories = traj
        out["all"] = {"trajectories": traj, "explained_variance_ratio": vr, "indices": idx}

    out["trajectories_full"] = trajectories
    return out


# RIDGE DECODER
def fit_ridge_decoder(trialized: TrializedData, cfg: ExploreConfig) -> Dict:
    """Decode hand_vel from smoothed rates, separately for active/passive blocks."""
    ti = trialized.trial_info
    has_bump = ti["ctr_hold_bump"].fillna(False).astype(bool).to_numpy()
    out = {}
    for name, mask in [("active", ~has_bump), ("passive", has_bump)]:
        X = trialized.rates[mask]
        Y = trialized.hand_vel[mask] if trialized.hand_vel is not None else None
        if Y is None or X.shape[0] == 0:
            continue
        lag_bins = int(round(cfg.decoder_lag_ms / trialized.bin_width_ms))
        if lag_bins > 0:
            X = X[:, :-lag_bins, :]
            Y = Y[:, lag_bins:, :]
        n_neurons = X.shape[2]
        Xf = X.reshape(-1, n_neurons)
        Yf = Y.reshape(-1, Y.shape[-1])
        valid = ~np.isnan(Yf).any(axis=1) & ~np.isnan(Xf).any(axis=1)
        Xf, Yf = Xf[valid], Yf[valid]
        cv = KFold(n_splits=cfg.decoder_cv_folds, shuffle=True, random_state=0)
        grid = GridSearchCV(Ridge(), {"alpha": cfg.decoder_alphas}, cv=cv, scoring="r2", n_jobs=-1)
        grid.fit(Xf, Yf)
        out[name] = {
            "r2": float(grid.best_score_),
            "best_alpha": float(grid.best_params_["alpha"]),
            "n_samples": int(Xf.shape[0]),
            "y_true": Yf,
            "y_pred": grid.predict(Xf),
        }
    return out


# PLOTTING
def plot_reach_paths(a2: Area2BumpDataset, cfg: ExploreConfig, out_path: Path):

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # color for each movement direction
    cmap = plt.cm.hsv
    direction_colors = {
        0: cmap(0/8),
        45: cmap(1/8),
        90: cmap(2/8),
        135: cmap(3/8),
        180: cmap(4/8),
        225: cmap(5/8),
        270: cmap(6/8),
        315: cmap(7/8),
    }

    for ax, bump, title in [
        (axes[0], False, "Active trials"),
        (axes[1], True, "Passive trials"),
    ]:

        ti = a2.dataset.trial_info

        mask = (
            (ti["ctr_hold_bump"].fillna(False) == bump)
            & (ti["split"] != "none")
        )

        for _, trial in ti[mask].iterrows():

            seg = a2.dataset.data.loc[
                trial["start_time"]:trial["end_time"]
            ]

            if "hand_pos" not in seg.columns.get_level_values(0):
                continue

            pos = seg["hand_pos"].to_numpy()

            direction = int(trial["cond_dir"])

            ax.plot(
                pos[:,0],
                pos[:,1],
                color=direction_colors[direction],
                alpha=0.7,
                lw=0.8,
            )

        ax.set_aspect("equal")
        ax.set_title(title)
        ax.set_xlabel("hand x (cm)")
        ax.set_ylabel("hand y (cm)")

    plt.tight_layout()
    plt.savefig(out_path, dpi=120)


def plot_single_neuron_psth(psth: PSTHResult, neuron_idx: int, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 5))
    cmap = plt.cm.tab20
    for i, label in enumerate(psth.condition_labels):
        ax.plot(psth.time_bins_ms, psth.psth[i, :, neuron_idx], label=label, color=cmap(i % 20), alpha=0.8)
    ax.axvline(0, color="k", linestyle="--", linewidth=0.7)
    ax.set_xlabel("time from move onset (ms)")
    ax.set_ylabel("firing rate (spikes/bin)")
    ax.set_title(f"PSTH — neuron #{neuron_idx}")
    ax.legend(fontsize=7, ncol=2, loc="upper right")
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)


def plot_raster(trialized: TrializedData, neuron_idx: int, max_trials: int, out_path: Path) -> None:
    n = min(max_trials, trialized.n_trials)
    fig, ax = plt.subplots(figsize=(10, 6))
    for i in range(n):
        sp = trialized.spikes[i, :, neuron_idx]
        times = np.where(sp > 0)[0]
        if len(times):
            ax.scatter(trialized.time_bins_ms[times], np.full_like(times, i),
                       s=2, color="black")
    ax.axvline(0, color="red", linestyle="--", linewidth=0.7)
    ax.set_xlabel("time from move onset (ms)")
    ax.set_ylabel("trial #")
    ax.set_title(f"Raster — neuron #{neuron_idx}")
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)


def plot_population_psth(psth: PSTHResult, out_path: Path) -> None:
    pop = psth.psth.mean(axis=2)   # (n_conditions, n_time)
    fig, ax = plt.subplots(figsize=(10, 5))
    cmap = plt.cm.tab20
    for i, label in enumerate(psth.condition_labels):
        ax.plot(psth.time_bins_ms, pop[i], label=label, color=cmap(i % 20), alpha=0.8)
    ax.axvline(0, color="k", linestyle="--", linewidth=0.7)
    ax.set_xlabel("time from move onset (ms)")
    ax.set_ylabel("population rate (spikes/bin)")
    ax.set_title("Population PSTH (mean over 65 neurons)")
    ax.legend(fontsize=7, ncol=2, loc="upper right")
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)


def plot_tuning_curve(tuning: Dict, neuron_idx: int, out_path: Path) -> None:
    dirs = tuning["directions_deg"]
    curve = tuning["tuning_curves"][neuron_idx]
    pref = tuning["preferred_direction_vm"][neuron_idx]
    fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(6, 6))
    theta = np.deg2rad(dirs)
    ax.plot(theta, curve, "o-", linewidth=1.5)
    if not np.isnan(pref):
        ax.plot(np.deg2rad(pref), curve.max(), "r*", markersize=15)
    ax.set_title(f"Tuning — neuron #{neuron_idx} (vm peak={pref:.0f}°)")
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)


def plot_pca_3d(traj: Dict, psth: PSTHResult, out_path: Path) -> None:
    fig = plt.figure(figsize=(10, 5))
    cmap = plt.cm.tab20
    for panel, (name, key) in enumerate([("active", "active"), ("passive", "passive")]):
        if key not in traj:
            continue
        ax = fig.add_subplot(1, 2, panel + 1, projection="3d")
        tr = traj[key]["trajectories"]
        idx = traj[key]["indices"]
        for i, gidx in enumerate(idx):
            ax.plot(tr[i, :, 0], tr[i, :, 1], tr[i, :, 2],
                    color=cmap(int(psth.condition_table.loc[gidx, "cond_dir"]) // 45 % 20),
                    alpha=0.8)
        ax.set_title(f"{name} — PCA trajectories")
        ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)


def plot_decoding_predictions(decoding: Dict, out_path: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, (key, axis_name) in zip(axes, [("x", 0), ("y", 1)]):
        for name, info in decoding.items():
            ax.scatter(info["y_true"][:, axis_name], info["y_pred"][:, axis_name],
                       s=2, alpha=0.3, label=f"{name} (R²={info['r2']:.2f})")
        lims = [ax.get_xlim(), ax.get_ylim()]
        lo, hi = min(lims[0][0], lims[1][0]), max(lims[0][1], lims[1][1])
        ax.plot([lo, hi], [lo, hi], "k--", linewidth=0.7)
        ax.set_xlabel("true")
        ax.set_ylabel("predicted")
        ax.set_title(f"hand_vel {key}")
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)


# CANONICAL HDF5 WRITER
def save_canonical_h5(out_path: Path, trialized: TrializedData, psth: PSTHResult,
                       traj: Dict, decoding: Dict, cfg: ExploreConfig) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(out_path, "w") as f:
        f.attrs["bin_width_ms"] = cfg.bin_width_ms
        f.attrs["align_field"] = cfg.align_field
        f.attrs["align_range_ms"] = cfg.align_range_ms
        f.attrs["kern_sd_ms"] = cfg.smooth_sd_ms
        f.attrs["n_neurons"] = trialized.n_neurons
        f.attrs["n_conditions"] = psth.n_conditions
        f.create_dataset("spikes", data=trialized.spikes)
        f.create_dataset("rates", data=trialized.rates)
        if trialized.hand_vel is not None:
            f.create_dataset("hand_vel", data=trialized.hand_vel)
        f.create_dataset("psth", data=psth.psth)
        f.create_dataset("time_bins_ms", data=psth.time_bins_ms)
        f.create_dataset("pca_trajectories", data=traj["trajectories_full"])
        cond = psth.condition_table
        f.create_dataset("condition_table", data=cond.to_numpy(dtype="S"))
        for name, info in decoding.items():
            grp = f.create_group(f"decoding/{name}")
            grp.attrs["r2"] = info["r2"]
            grp.attrs["best_alpha"] = info["best_alpha"]
            grp.create_dataset("y_true", data=info["y_true"])
            grp.create_dataset("y_pred", data=info["y_pred"])



# PIPELINE DRIVER


if __name__ == "__main__":
    cfg = ExploreConfig()
    paths = cfg.resolved_paths()
    summary = {"config": cfg.__dict__}

    # Load & preprocess
    a2 = Area2BumpDataset(cfg)
    summary["dataset"] = {"n_trials": a2.n_trials, "n_neurons": a2.n_neurons,
                          "bin_width_ms": a2.dataset.bin_width}

    # Trial-type table
    trial_type_table = get_trial_type_table(a2.dataset.trial_info)
    trial_type_table.to_csv(paths["out_dir"] / "trial_type_table.csv", index=False)
    summary["trial_types"] = {"n": int(len(trial_type_table)), "active": int((~trial_type_table["ctr_hold_bump"]).sum()),
                              "passive": int(trial_type_table["ctr_hold_bump"].sum())}

    # Reach paths
    plot_reach_paths(a2, cfg, paths["fig_dir"] / "01_reach_paths.png")

    # Trialize
    trialized = trialize_area2_bump(a2, cfg)
    summary["trialized"] = {"n_trials": trialized.n_trials, "n_time": trialized.n_time,
                            "n_neurons": trialized.n_neurons,
                            "spikes_shape": list(trialized.spikes.shape),
                            "rates_shape": list(trialized.rates.shape)}

    # PSTH
    psth = compute_psth(trialized, cfg)
    psth.condition_table.to_csv(paths["out_dir"] / "psth_condition_table.csv", index=False)
    summary["psth"] = {"shape": list(psth.psth.shape),
                       "n_conditions": psth.n_conditions, "kern_sd_ms": psth.kern_sd_ms}

    plot_single_neuron_psth(psth, cfg.neuron_to_plot, paths["fig_dir"] / "02_psth_neuron.png")
    plot_raster(trialized, cfg.neuron_to_plot, cfg.max_raster_trials, paths["fig_dir"] / "03_raster_neuron.png")
    plot_population_psth(psth, paths["fig_dir"] / "04_population_psth.png")

    # Tuning
    tuning = compute_tuning_curves(psth, cfg)
    tuning_df = pd.DataFrame({
        "neuron_idx": np.arange(tuning["tuning_curves"].shape[0]),
        "preferred_argmax_deg": tuning["preferred_direction_argmax"],
        "preferred_vm_deg": tuning["preferred_direction_vm"],
        "vm_kappa": tuning["von_mises_kappa"],
        "fit_success": tuning["fit_success"],
    })
    tuning_df.to_csv(paths["out_dir"] / "tuning_curves.csv", index=False)
    summary["tuning"] = {
        "n_neurons": int(tuning["tuning_curves"].shape[0]),
        "n_fit_success": int(tuning["fit_success"].sum()),
        "median_kappa": float(np.nanmedian(tuning["von_mises_kappa"])),
        "mean_abs_argmax_vm_diff_deg": float(np.nanmean(
            np.abs(tuning["preferred_direction_argmax"] - tuning["preferred_direction_vm"])
        )),
    }
    plot_tuning_curve(tuning, cfg.neuron_to_plot, paths["fig_dir"] / "05_tuning_neuron.png")

    # PCA trajectories
    traj = compute_pca_trajectories(psth, cfg)
    summary["pca"] = {
        "n_components": cfg.pca_n_components,
        "active_evr": traj.get("active", {}).get("explained_variance_ratio", None).tolist() if "active" in traj else None,
        "passive_evr": traj.get("passive", {}).get("explained_variance_ratio", None).tolist() if "passive" in traj else None,
    }
    plot_pca_3d(traj, psth, paths["fig_dir"] / "06_pca_trajectories.png")

    # Decoding
    decoding = fit_ridge_decoder(trialized, cfg)
    summary["decoding"] = {k: {"r2": v["r2"], "best_alpha": v["best_alpha"],
                                "n_samples": v["n_samples"]} for k, v in decoding.items()}
    plot_decoding_predictions(decoding, paths["fig_dir"] / "07_decoding.png")

    # Save canonical H5
    save_canonical_h5(paths["canonical_h5"], trialized, psth, traj, decoding, cfg)
    summary["canonical_h5"] = str(paths["canonical_h5"])
    summary["canonical_h5_size_mb"] = round(os.path.getsize(paths["canonical_h5"]) / 1e6, 2)

    # Write summary JSON
    with open(paths["summary_json"], "w") as f:
        json.dump(summary, f, indent=2, default=str)

    print(f"[explore] OK  trials={trialized.n_trials} neurons={trialized.n_neurons} "
          f"conditions={psth.n_conditions} r2_active={summary['decoding'].get('active', {}).get('r2', float('nan')):.3f} "
          f"-> {paths['summary_json']}")


[explore] OK  trials=364 neurons=65 conditions=16 r2_active=0.696 -> /kaggle/working/area2_explore_summary.json
